In [5]:
import pandas as pd
import numpy as np
import pygeohash as pgh
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy.optimize import minimize

In [6]:
train = pd.read_csv("dataset/train.csv")
test = pd.read_csv("dataset/test.csv")


In [7]:
def decode_geohash(gh):
    try:
        lat, lon = pgh.decode(gh)
        return pd.Series([lat, lon])
    except:
        return pd.Series([np.nan, np.nan])

In [8]:
train[['latitude', 'longitude']] = train['geohash'].apply(decode_geohash)
test[['latitude', 'longitude']] = test['geohash'].apply(decode_geohash)

In [9]:
for df in [train, test]:
    # Fill missing timestamps to prevent the NaN to int conversion error
    df['timestamp'] = df['timestamp'].fillna('00:00')

    df['hour'] = df['timestamp'].str.split(':').str[0].astype(int)
    df['minute'] = df['timestamp'].str.split(':').str[1].astype(int)
    df['minutes_since_midnight'] = df['hour'] * 60 + df['minute']
    df['time_sin'] = np.sin(2 * np.pi * df['minutes_since_midnight'] / 1440)
    df['time_cos'] = np.cos(2 * np.pi * df['minutes_since_midnight'] / 1440)

In [10]:
for col in ['Temperature', 'latitude', 'longitude']:
    train[col] = train[col].fillna(train[col].median())
    test[col] = test[col].fillna(train[col].median())

In [11]:
for col in ['RoadType', 'Weather']:
    train[col] = train[col].fillna('Unknown')
    test[col] = test[col].fillna('Unknown')

In [12]:
y = train['demand']
X = train.drop(columns=['Index', 'demand', 'timestamp'])
X_test = test.drop(columns=['Index', 'timestamp'])

In [13]:
categorical_features = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

In [14]:
for col in categorical_features:
    X[col] = X[col].astype('object').fillna('Unknown').astype(str).astype('category')
    X_test[col] = X_test[col].astype('object').fillna('Unknown').astype(str).astype('category')

In [15]:
cat_model = CatBoostRegressor(iterations=12000, depth=8, learning_rate=0.03, loss_function='RMSE', early_stopping_rounds=4000, l2_leaf_reg=5, verbose=500)

xgb_model = XGBRegressor(n_estimators=3000, max_depth=6, learning_rate=0.03, enable_categorical=True, tree_method='hist', early_stopping_rounds=50, reg_lambda=3, verbosity=2)

lgb_model = LGBMRegressor(n_estimators=12000, max_depth=8, learning_rate=0.03, early_stopping_rounds=4000, reg_lambda=3, verbose=1)

In [16]:
print("Training models on 80% split...")
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
cat_model.fit(X_train, y_train, eval_set=(X_valid, y_valid), cat_features=categorical_features)
xgb_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)])
lgb_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)])

cat_val_preds = cat_model.predict(X_valid)
xgb_val_preds = xgb_model.predict(X_valid)
lgb_val_preds = lgb_model.predict(X_valid)

# Define the function we want to minimize (negative R2 score)
def objective_func(weights):
    blended_preds = (weights[0] * cat_val_preds +
                     weights[1] * xgb_val_preds +
                     weights[2] * lgb_val_preds)
    return -r2_score(y_valid, blended_preds)

# Start with an equal average
initial_weights = [1/3, 1/3, 1/3]

# Keep weights between 0 and 1
bounds = ((0, 1), (0, 1), (0, 1))

# Ensure the weights always add up to exactly 1.0
constraints = {'type': 'eq', 'fun': lambda w: sum(w) - 1}

# Calculate the best weights
result = minimize(objective_func, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)
best_weights = result.x

print(f"Optimal Weights - CatBoost: {best_weights[0]:.3f}, XGBoost: {best_weights[1]:.3f}, LightGBM: {best_weights[2]:.3f}")

# Check the new R2 score
weighted_val_preds = (best_weights[0] * cat_val_preds +
                      best_weights[1] * xgb_val_preds +
                      best_weights[2] * lgb_val_preds)
print("Weighted Validation R2 Score =", r2_score(y_valid, weighted_val_preds))

Training models on 80% split...
0:	learn: 0.1389258	test: 0.1389142	best: 0.1389142 (0)	total: 76.1ms	remaining: 15m 13s
500:	learn: 0.0351408	test: 0.0357007	best: 0.0357007 (500)	total: 9.93s	remaining: 3m 47s
1000:	learn: 0.0315461	test: 0.0331145	best: 0.0331145 (1000)	total: 20.3s	remaining: 3m 42s
1500:	learn: 0.0294989	test: 0.0319026	best: 0.0319026 (1500)	total: 32.5s	remaining: 3m 47s
2000:	learn: 0.0280067	test: 0.0312243	best: 0.0312241 (1998)	total: 46s	remaining: 3m 49s
2500:	learn: 0.0267659	test: 0.0307609	best: 0.0307609 (2500)	total: 58.1s	remaining: 3m 40s
3000:	learn: 0.0257970	test: 0.0304388	best: 0.0304385 (2999)	total: 1m 9s	remaining: 3m 27s
3500:	learn: 0.0250170	test: 0.0302497	best: 0.0302497 (3494)	total: 1m 20s	remaining: 3m 14s
4000:	learn: 0.0243521	test: 0.0300848	best: 0.0300848 (4000)	total: 1m 31s	remaining: 3m 2s
4500:	learn: 0.0237828	test: 0.0299994	best: 0.0299984 (4487)	total: 1m 41s	remaining: 2m 49s
5000:	learn: 0.0232802	test: 0.0299077	best:

In [17]:
final_cat_model = CatBoostRegressor(iterations=12000, depth=8, learning_rate=0.03, loss_function='RMSE', l2_leaf_reg=5, verbose=500)
final_xgb_model = XGBRegressor(n_estimators=12000, max_depth=6, learning_rate=0.03, enable_categorical=True, tree_method='hist', reg_lambda=3, verbosity=0)
final_lgb_model = LGBMRegressor(n_estimators=12000, max_depth=8, learning_rate=0.03, reg_lambda=3, verbose=-1)

print("Training CatBoost on 100% data...")
final_cat_model.fit(X, y, cat_features=categorical_features)

print("Training XGBoost on 100% data...")
final_xgb_model.fit(X, y)

print("Training LightGBM on 100% data...")
final_lgb_model.fit(X, y)

Training CatBoost on 100% data...
0:	learn: 0.1389277	total: 17.6ms	remaining: 52.6s
500:	learn: 0.0389236	total: 7.52s	remaining: 37.5s
1000:	learn: 0.0352717	total: 15.3s	remaining: 30.5s
1500:	learn: 0.0332803	total: 23s	remaining: 22.9s
2000:	learn: 0.0318696	total: 30.9s	remaining: 15.4s
2500:	learn: 0.0307825	total: 38.7s	remaining: 7.73s
2999:	learn: 0.0299054	total: 46.7s	remaining: 0us
Training XGBoost on 100% data...
Training LightGBM on 100% data...


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,6
,learning_rate,0.03
,n_estimators,3000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [18]:
cat_train_preds = cat_model.predict(X)
xgb_train_preds = xgb_model.predict(X)
lgb_train_preds = lgb_model.predict(X)

In [23]:
cat_preds = final_cat_model.predict(X_test)
xgb_preds = final_xgb_model.predict(X_test)
lgb_preds = final_lgb_model.predict(X_test)

In [24]:
final_predictions = (best_weights[0] * cat_preds +
                     best_weights[1] * xgb_preds +
                     best_weights[2] * lgb_preds)

In [25]:
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': final_predictions
})
submission.to_csv('ensemble_submission15.csv', index=False)
print("Ensemble submission saved as 'ensemble_submission15.csv'!")

Ensemble submission saved as 'ensemble_submission15.csv'!
